In [3]:
# =========================================================
# SETUP
# =========================================================

import sys, subprocess, os, gc, json, random, shutil, warnings
warnings.filterwarnings("ignore")

def safe_import(package_name, import_name=None):
    try:
        __import__(import_name or package_name)
        print(f"✓ {package_name} installed")
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        print(f"✓ Installed {package_name}")

safe_import("transformers")
safe_import("datasets")
safe_import("jiwer")
safe_import("accelerate")

import numpy as np
import pandas as pd
import torch

from dataclasses import dataclass
from typing import Union

from datasets import Dataset, DatasetDict, Audio, load_from_disk

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

from jiwer import wer

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

✓ transformers installed
✓ datasets installed
✓ jiwer installed
✓ accelerate installed
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [6]:
# =========================================================
# CONFIG
# =========================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_DATASET_DIR = "/kaggle/input/datasets/obaidah/asr-nn"

MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

WORK_DIR = "/kaggle/working"

RUN_DIR = f"{WORK_DIR}/wav2vec2_prod_run"
FINAL_DIR = f"{WORK_DIR}/wav2vec2_prod_final"
PROC_BASE = f"{WORK_DIR}/wav2vec2_processed"

PRED_PATH = f"{WORK_DIR}/wav2vec2_prod_predictions.csv"
SUMMARY_PATH = f"{WORK_DIR}/wav2vec2_prod_summary.json"

# حجم الداتا في التجربة
TRAIN_LIMIT = 12000
VAL_LIMIT   = 1500
TEST_LIMIT  = 1500

# Training
MAX_STEPS = 1200          
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 10

BATCH_SIZE = 4
GRAD_ACCUM = 2            # effective batch = 8

LR = 1e-5                 # أفضل من 3e-5 لو أنت already قريب من 24 WER
WARMUP_STEPS = 100

RESUME = True             # مهم جدًا
USE_SAVED_PROCESSED = True

print("Config ready")

Config ready


In [22]:
# =========================================================
# STAGE 2 CONFIG: 40K + RESUME
# =========================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_DATASET_DIR = "/kaggle/input/datasets/obaidah/asr-nn"

MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

WORK_DIR = "/kaggle/working"

RUN_DIR = f"{WORK_DIR}/wav2vec2_prod_run"
FINAL_DIR = f"{WORK_DIR}/wav2vec2_prod_final"
PROC_BASE = f"{WORK_DIR}/wav2vec2_processed_40k"

PRED_PATH = f"{WORK_DIR}/wav2vec2_prod_predictions_stage2.csv"
SUMMARY_PATH = f"{WORK_DIR}/wav2vec2_prod_summary_stage2.json"

TRAIN_LIMIT = 40000
VAL_LIMIT   = 3000
TEST_LIMIT  = 3000

MAX_STEPS = 4000

EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 10

BATCH_SIZE = 4
GRAD_ACCUM = 2

LR = 3e-6
WARMUP_STEPS = 50

RESUME = True
USE_SAVED_PROCESSED = False

print("Stage 2 config ready")

Stage 2 config ready


In [8]:
# =========================================================
# LOAD DATAFRAMES
# =========================================================

train_csv = os.path.join(BASE_DATASET_DIR, "train_ready.csv")
val_csv   = os.path.join(BASE_DATASET_DIR, "val_ready.csv")
test_csv  = os.path.join(BASE_DATASET_DIR, "test_ready.csv")

if not os.path.exists(train_csv):
    train_csv = os.path.join(BASE_DATASET_DIR, "train.csv")
if not os.path.exists(val_csv):
    val_csv = os.path.join(BASE_DATASET_DIR, "val.csv")
if not os.path.exists(test_csv):
    test_csv = os.path.join(BASE_DATASET_DIR, "test.csv")

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
test_df  = pd.read_csv(test_csv)

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df.columns)

(62973, 5) (7872, 5) (7872, 5)
Index(['text', 'file_name', 'audio_path', 'duration', 'resampled_path'], dtype='object')


In [11]:
# =========================================================
# FIND REAL AUDIO PATHS
# =========================================================

audio_exts = (".wav", ".mp3", ".flac", ".m4a", ".ogg")
real_audio_map = {}

for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith(audio_exts):
            if f not in real_audio_map:
                real_audio_map[f] = os.path.join(root, f)

print("Audio files found:", len(real_audio_map))


def prepare_df(df):
    df = df.copy()

    if "normalized_text" in df.columns:
        text_col = "normalized_text"
    elif "text" in df.columns:
        text_col = "text"
    elif "sentence" in df.columns:
        text_col = "sentence"
    else:
        raise ValueError("No text column found")

    if "file_name" not in df.columns:
        raise ValueError("No file_name column found")

    df["audio_path_real"] = df["file_name"].astype(str).map(real_audio_map)
    df = df[df["audio_path_real"].notna()].copy()

    df["text_final"] = df[text_col].astype(str).str.strip()
    df["text_final"] = df["text_final"].str.replace(r"\s+", " ", regex=True)

    df = df[df["text_final"].str.len() > 0].reset_index(drop=True)

    return df[["file_name", "audio_path_real", "text_final"]]


train_df = prepare_df(train_df)
val_df   = prepare_df(val_df)
test_df  = prepare_df(test_df)

print("Prepared train:", len(train_df))
print("Prepared val:", len(val_df))
print("Prepared test:", len(test_df))

Audio files found: 78717
Prepared train: 62973
Prepared val: 7872
Prepared test: 7872


In [12]:
# =========================================================
# SAMPLE 40K
# =========================================================

train_df_run = train_df.sample(
    min(TRAIN_LIMIT, len(train_df)),
    random_state=SEED
).reset_index(drop=True)

val_df_run = val_df.sample(
    min(VAL_LIMIT, len(val_df)),
    random_state=SEED
).reset_index(drop=True)

test_df_run = test_df.sample(
    min(TEST_LIMIT, len(test_df)),
    random_state=SEED
).reset_index(drop=True)

print("RUN train:", len(train_df_run))
print("RUN val:", len(val_df_run))
print("RUN test:", len(test_df_run))

RUN train: 40000
RUN val: 3000
RUN test: 3000


In [13]:
# =========================================================
# LOAD MODEL + PROCESSOR
# =========================================================

processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)

model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Loaded:", MODEL_ID)
print("Device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loaded: jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Device: cuda:0


In [14]:
# =========================================================
# BUILD PROCESSED DATASET 40K
# =========================================================

from datasets import Dataset, DatasetDict, Audio, load_from_disk


def make_hf_dataset(df):
    return Dataset.from_pandas(
        df[["audio_path_real", "text_final"]].rename(
            columns={
                "audio_path_real": "audio",
                "text_final": "sentence"
            }
        ),
        preserve_index=False
    )


def prepare_batch(batch):
    audio = batch["audio"]

    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    batch["labels"] = processor(text=batch["sentence"]).input_ids

    return batch


if USE_SAVED_PROCESSED and os.path.exists(f"{PROC_BASE}/train"):
    print("Loading processed 40k dataset from disk...")

    train_proc = load_from_disk(f"{PROC_BASE}/train")
    val_proc   = load_from_disk(f"{PROC_BASE}/validation")
    test_proc  = load_from_disk(f"{PROC_BASE}/test")

else:
    print("Creating processed 40k dataset...")

    dataset = DatasetDict({
        "train": make_hf_dataset(train_df_run),
        "validation": make_hf_dataset(val_df_run),
        "test": make_hf_dataset(test_df_run)
    })

    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

    train_proc = dataset["train"].map(
        prepare_batch,
        remove_columns=dataset["train"].column_names,
        desc="Preparing train 40k"
    )

    val_proc = dataset["validation"].map(
        prepare_batch,
        remove_columns=dataset["validation"].column_names,
        desc="Preparing validation"
    )

    test_proc = dataset["test"].map(
        prepare_batch,
        remove_columns=dataset["test"].column_names,
        desc="Preparing test"
    )

    os.makedirs(PROC_BASE, exist_ok=True)

    # train_proc.save_to_disk(f"{PROC_BASE}/train")
    # val_proc.save_to_disk(f"{PROC_BASE}/validation")
    # test_proc.save_to_disk(f"{PROC_BASE}/test")

    print("Saved processed 40k dataset to:", PROC_BASE)

print(train_proc)
print(val_proc)
print(test_proc)

Creating processed 40k dataset...


Preparing train 40k:   0%|          | 0/40000 [00:00<?, ? examples/s]

Preparing validation:   0%|          | 0/3000 [00:00<?, ? examples/s]

Preparing test:   0%|          | 0/3000 [00:00<?, ? examples/s]

Saved processed 40k dataset to: /kaggle/working/wav2vec2_processed_40k
Dataset({
    features: ['input_values', 'labels'],
    num_rows: 40000
})
Dataset({
    features: ['input_values', 'labels'],
    num_rows: 3000
})
Dataset({
    features: ['input_values', 'labels'],
    num_rows: 3000
})


In [15]:
# =========================================================
# DATA COLLATOR
# =========================================================

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

print("Data collator ready")

Data collator ready


In [16]:
# =========================================================
# COMPUTE WER
# =========================================================

def normalize_text_list(texts):
    return [" ".join(str(x).strip().split()) for x in texts]


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred_texts = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    ref_texts = processor.batch_decode(label_ids, group_tokens=False)

    pred_texts = normalize_text_list(pred_texts)
    ref_texts  = normalize_text_list(ref_texts)

    return {"wer": wer(ref_texts, pred_texts)}

In [23]:
# =========================================================
# TRAINING ARGUMENTS STAGE 2
# =========================================================

training_args = TrainingArguments(
    output_dir=RUN_DIR,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,

    max_steps=MAX_STEPS,

    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",

    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=0,

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    remove_unused_columns=False,

    group_by_length=False,
    length_column_name="input_length",

    push_to_hub=False
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=100,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False

In [10]:
# # =========================================================
# # ADD INPUT LENGTH
# # =========================================================

# def add_input_length(batch):
#     batch["input_length"] = len(batch["input_values"])
#     return batch

# if "input_length" not in train_proc.column_names:
#     train_proc = train_proc.map(add_input_length, desc="Adding train lengths")

# if "input_length" not in val_proc.column_names:
#     val_proc = val_proc.map(add_input_length, desc="Adding val lengths")

# if "input_length" not in test_proc.column_names:
#     test_proc = test_proc.map(add_input_length, desc="Adding test lengths")

# print(train_proc.column_names)

In [24]:
# =========================================================
# TRAINER
# =========================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_proc,
    eval_dataset=val_proc,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=4,
            early_stopping_threshold=0.001
        )
    ]
)

print("Trainer ready")

Trainer ready


In [25]:
# =========================================================
# RESUME CHECKPOINT DETECTION
# =========================================================

def get_last_checkpoint(run_dir):
    if not os.path.exists(run_dir):
        return None

    checkpoints = [
        os.path.join(run_dir, d)
        for d in os.listdir(run_dir)
        if d.startswith("checkpoint-")
    ]

    if len(checkpoints) == 0:
        return None

    checkpoints = sorted(
        checkpoints,
        key=lambda x: int(x.split("-")[-1])
    )

    return checkpoints[-1]


last_ckpt = get_last_checkpoint(RUN_DIR)

print("Last checkpoint:", last_ckpt)

Last checkpoint: /kaggle/working/wav2vec2_prod_run/checkpoint-1600


In [20]:
# =========================================================
# TRAIN
# =========================================================

if RESUME and last_ckpt is not None:
    print("Resuming from:", last_ckpt)
    train_output = trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("Starting fresh training")
    train_output = trainer.train()

print(train_output)

Resuming from: /kaggle/working/wav2vec2_prod_run/checkpoint-1200


Step,Training Loss,Validation Loss,Wer
1300,1.140002,0.333108,0.155840
1400,1.294840,0.327805,0.155169
1500,1.193039,0.324017,0.154193
1600,1.222336,0.322655,0.153766


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1600, training_loss=0.29873511552810667, metrics={'train_runtime': 2685.29, 'train_samples_per_second': 14.896, 'train_steps_per_second': 0.931, 'total_flos': 3.672883292744516e+18, 'train_loss': 0.29873511552810667, 'epoch': 0.64})


In [26]:
# =========================================================
# TRAIN
# =========================================================

if RESUME and last_ckpt is not None:
    print("Resuming from:", last_ckpt)
    train_output = trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("Starting fresh training")
    train_output = trainer.train()

print(train_output)

Resuming from: /kaggle/working/wav2vec2_prod_run/checkpoint-1600


Step,Training Loss,Validation Loss,Wer
1700,0.923984,0.321036,0.154742
1800,1.232182,0.316057,0.155901
1900,1.376516,0.314758,0.153766
2000,1.163202,0.311089,0.152668


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=0.2443341064453125, metrics={'train_runtime': 2646.7265, 'train_samples_per_second': 24.181, 'train_steps_per_second': 1.511, 'total_flos': 4.934728784422508e+18, 'train_loss': 0.2443341064453125, 'epoch': 0.8})


In [21]:
# =========================================================
# VALIDATION + TEST EVALUATION
# =========================================================

val_metrics = trainer.evaluate(val_proc)
test_metrics = trainer.evaluate(test_proc)

print("Validation metrics:", val_metrics)
print("Test metrics:", test_metrics)

Validation metrics: {'eval_loss': 0.33553987741470337, 'eval_wer': 0.15559621835925588, 'eval_runtime': 287.9045, 'eval_samples_per_second': 10.42, 'eval_steps_per_second': 1.303, 'epoch': 0.64}
Test metrics: {'eval_loss': 0.3116284906864166, 'eval_wer': 0.15543818706626544, 'eval_runtime': 287.5752, 'eval_samples_per_second': 10.432, 'eval_steps_per_second': 1.304, 'epoch': 0.64}


In [27]:
# =========================================================
# VALIDATION + TEST EVALUATION
# =========================================================

val_metrics = trainer.evaluate(val_proc)
test_metrics = trainer.evaluate(test_proc)

print("Validation metrics:", val_metrics)
print("Test metrics:", test_metrics)

Validation metrics: {'eval_loss': 0.33553987741470337, 'eval_wer': 0.15559621835925588, 'eval_runtime': 287.8421, 'eval_samples_per_second': 10.422, 'eval_steps_per_second': 1.303, 'epoch': 0.8}
Test metrics: {'eval_loss': 0.3116284906864166, 'eval_wer': 0.15543818706626544, 'eval_runtime': 286.8638, 'eval_samples_per_second': 10.458, 'eval_steps_per_second': 1.307, 'epoch': 0.8}


In [28]:
# =========================================================
# SAVE FINAL MODEL
# =========================================================

trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)

print("Saved final model to:", FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model to: /kaggle/working/wav2vec2_prod_final


In [29]:
# =========================================================
# SAVE PREDICTIONS
# =========================================================

pred_out = trainer.predict(test_proc)

pred_ids = np.argmax(pred_out.predictions, axis=-1)
pred_texts = processor.batch_decode(pred_ids)

label_ids = pred_out.label_ids.copy()
label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

ref_texts = processor.batch_decode(label_ids, group_tokens=False)

pred_texts = normalize_text_list(pred_texts)
ref_texts  = normalize_text_list(ref_texts)

pred_df = pd.DataFrame({
    "reference": ref_texts,
    "prediction": pred_texts
})

pred_df.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")

print("Saved predictions to:", PRED_PATH)
display(pred_df.head(20))

Saved predictions to: /kaggle/working/wav2vec2_prod_predictions_stage2.csv


,reference,prediction
0,من ذا الذي يقرض الله قرضا حسنا فيضاعفه له وله ...,من ذا الذي يقرض الله قرضا حسنا فيضاعفه له وله ...
1,لم يعد عندنا اي سكر,لم يعد عننا اي سكر
2,حقا <unk>,حقا
3,انا ذاهب,انا ذاهب
4,الا ابكي علي صخر وصخر ثمالنا,الا ابكي علي صخر وسخر ثمالنا
5,قالوا بل وجدنا اباءنا كذلك يفعلون,قالوا بل وجدنا اباءنا كذلك يفعلون
6,ولفقيه واحد اشد علي الشيطان من الف عابد,ولفقيه واحد اشد علي الشيطان من الف عابد
7,ومن الناس من يجادل في الله بغير علم ولا هدي ول...,ومن الناس من يجادل في الله بغير علم ولا هدي ول...
8,اني في مشكلة الان,اني في مشكلة الان
9,ايمكنك ملاحظة الفرق<unk>,ايمكنك ملاحظة الفرق


In [30]:
# =========================================================
# SAVE SUMMARY
# =========================================================

summary = {
    "model_id": MODEL_ID,
    "train_samples": len(train_df_run),
    "val_samples": len(val_df_run),
    "test_samples": len(test_df_run),
    "max_steps": MAX_STEPS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "learning_rate": LR,
    "warmup_steps": WARMUP_STEPS,
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "train_output": {
        "global_step": train_output.global_step,
        "training_loss": float(train_output.training_loss),
        "metrics": train_output.metrics
    }
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved summary to:", SUMMARY_PATH)

Saved summary to: /kaggle/working/wav2vec2_prod_summary_stage2.json


In [31]:
# =========================================================
# CREATE LIGHT BUNDLE
# =========================================================

BUNDLE_DIR = f"{WORK_DIR}/final_bundle_light"
ZIP_PATH = f"{WORK_DIR}/wav2vec2_light_bundle.zip"

if os.path.exists(BUNDLE_DIR):
    shutil.rmtree(BUNDLE_DIR)

os.makedirs(BUNDLE_DIR, exist_ok=True)

if os.path.exists(FINAL_DIR):
    shutil.copytree(FINAL_DIR, f"{BUNDLE_DIR}/model_final")
    print("✔ model_final copied")

if os.path.exists(PRED_PATH):
    shutil.copy(PRED_PATH, f"{BUNDLE_DIR}/predictions.csv")
    print("✔ predictions copied")

if os.path.exists(SUMMARY_PATH):
    shutil.copy(SUMMARY_PATH, f"{BUNDLE_DIR}/summary.json")
    print("✔ summary copied")

with open(f"{BUNDLE_DIR}/training_args.json", "w") as f:
    json.dump(training_args.to_dict(), f, indent=2)

model_info = {
    "model": "Wav2Vec2 Arabic Fine-tuned",
    "base_model": MODEL_ID,
    "final_dir": FINAL_DIR,
    "best_metric": trainer.state.best_metric,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "global_step": trainer.state.global_step
}

with open(f"{BUNDLE_DIR}/model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

shutil.make_archive(
    base_name=ZIP_PATH.replace(".zip", ""),
    format="zip",
    root_dir=BUNDLE_DIR
)

print("🔥 Bundle ready:", ZIP_PATH)

✔ model_final copied
✔ predictions copied
✔ summary copied
🔥 Bundle ready: /kaggle/working/wav2vec2_light_bundle.zip


In [32]:
# =========================================================
# SHOW FILES
# =========================================================

from IPython.display import FileLink

def sizeof_fmt(num):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if num < 1024:
            return f"{num:.2f} {unit}"
        num /= 1024

print("===== /kaggle/working =====")

for root, dirs, files in os.walk(WORK_DIR):
    level = root.replace(WORK_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 2 * (level + 1)

    for f in files:
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        print(f"{subindent}{f} ({sizeof_fmt(size)})")

print("\nZIP:", ZIP_PATH)
print("Size:", sizeof_fmt(os.path.getsize(ZIP_PATH)))

FileLink(ZIP_PATH)

===== /kaggle/working =====
working/
  wav2vec2_prod_summary_stage2.json (1.01 KB)
  wav2vec2_light_bundle.zip (1.09 GB)
  wav2vec2_prod_predictions_stage2.csv (295.45 KB)
  wav2vec2_processed_40k/
  final_bundle_light/
    summary.json (1.01 KB)
    model_info.json (302.00 B)
    predictions.csv (295.45 KB)
    training_args.json (3.32 KB)
    model_final/
      model.safetensors (1.18 GB)
      processor_config.json (299.00 B)
      vocab.json (612.00 B)
      training_args.bin (5.08 KB)
      tokenizer_config.json (1.26 KB)
      config.json (2.21 KB)
  wav2vec2_prod_final/
    model.safetensors (1.18 GB)
    processor_config.json (299.00 B)
    vocab.json (612.00 B)
    training_args.bin (5.08 KB)
    tokenizer_config.json (1.26 KB)
    config.json (2.21 KB)
  .virtual_documents/
    __notebook_source__.ipynb (18.32 KB)
  wav2vec2_prod_run/
    checkpoint-1200/
      scheduler.pt (1.43 KB)
      optimizer.pt (2.32 GB)
      model.safetensors (1.18 GB)
      processor_config.json (2

/kaggle/working/wav2vec2_light_bundle.zip